# Run this cell first

In [ ]:
# this code enables the automated feedback. If you remove this, you won't get any feedback
# so don't delete this cell!
try:
  import AutoFeedback
except (ModuleNotFoundError, ImportError):
  %pip install AutoFeedback
  import AutoFeedback

try:
  from testsrc import test_main
except (ModuleNotFoundError, ImportError):
  %pip install "git+https://github.com/autofeedback-exercises/exercises.git#subdirectory=MTH2100/numerical_analysis/errors_and_floating_point"
  from testsrc import test_main

def runtest(tlist):
  import unittest
  from contextlib import redirect_stderr
  from os import devnull
  with redirect_stderr(open(devnull, 'w')):
    suite = unittest.TestSuite()
    for tname in tlist:
      suite.addTest(eval(f"test_main.UnitTests.{tname}"))
    runner = unittest.TextTestRunner()
    try:
      runner.run(suite)
    except (AssertionError, ImportError):
      pass

# 1. Numerical Errors and Floating-Point Arithmetic

## Why this matters

Every number your computer stores is an approximation. Every arithmetic operation introduces
a small error. When you run a simulation for thousands of time steps these small errors
accumulate – and can completely corrupt your results if you are not careful.

This notebook explores where numerical errors come from, how to measure them, and how to
recognise when they are causing problems. These ideas underpin every other technique in
this numerical-analysis pathway.

**Learning objectives**
- Understand how floating-point numbers are stored and why this limits precision
- Distinguish *truncation error* (from approximating infinite processes) from *rounding
  error* (from finite precision arithmetic)
- Implement forward, backward, and central difference approximations and understand their
  relative accuracy
- Recognise catastrophic cancellation and know how to avoid it

<div style="background:#e8f4f8;border-left:4px solid #2196F3;padding:16px;margin:16px 0;border-radius:4px;width:100%;box-sizing:border-box;overflow-wrap:break-word;">

**📹 VIDEO: Floating-Point Numbers and Numerical Error**

*What this video covers:* Why computers cannot store most real numbers exactly; the IEEE 754 standard; machine epsilon; the difference between truncation error and rounding error; absolute vs relative error.

*(Embed the video here using an `<iframe>` or `IPython.display.Video`)*
</div>


## 1.1 Floating-Point Representation

Computers store real numbers in **IEEE 754 binary floating-point** format. A standard
64-bit `float64` uses:
- 1 **sign** bit
- 11 **exponent** bits
- 52 **mantissa** bits

Only numbers of the form $m \times 2^e$ (with a 52-bit integer $m$) can be stored exactly.
Most decimal fractions — including $0.1$ — cannot.


In [ ]:
print(0.1+0.2)

This has implications for certain logical operations. For instance, if you're trying to calculate when a function gives an answer of 0.3, you can't check that with a simple equivalence check: normally to compare if two numbers are equal we use `a == b`, but because of the inexactness of the way numbers are stored, this won't always work:

In [ ]:
a = 0.1 + 0.2
b = 0.3
if a == b:
    print("they're equal!")
else:
    print("0.1 + 0.2 does not equal 0.3!")

One way to solve the problem is to check that the two numbers are 'sufficiently' close together:

In [ ]:
if (abs(a-b) < 1e-6):
    print("they're equal!")

Thankfully, it's a common enough problem that numpy has a built in check for numbers that are close enough together:


In [ ]:
import numpy as np

print(np.isclose(0.1+0.2, 0.3))

## 1.2 Machine precision

Because of the way numbers are stored, there is a smallest possible change to a number that can be represented on a computer. Notice, that's not the same thing as a smallest possible number (although there is a limit there as well). This has to do with detecting changes to numbers. The **machine epsilon** $\varepsilon_{\mathrm{mach}}$ is the smallest number such that
$1 + \varepsilon_{\mathrm{mach}} > 1$ in floating-point arithmetic. It quantifies the relative precision of the number system. For '64 bit floats' – the standard number format used by python – the machine epsilon is 

$$\varepsilon_{\mathrm{mach}} \approx 2.22 \times 10^{-16}.$$

$\varepsilon_{\mathrm{mach}}$ puts a limit on how 'exact' an answer we can expect. When we printed `0.1+0.2` earlier on, notice that the result `0.30000000000000004` is exact up to the **16th** significant figure.

In [ ]:
eps = np.finfo(float).eps
print(eps)

# adding eps to 1 will just about register
print(1+eps)

# but adding something smaller than eps (e.g. eps/2) does not register
print(1+eps/2)


## 1.3 Absolute and relative error

The absolute error is the modulus of the difference between an exact and approximate value. E.G a well known approximation is $\pi \approx \frac{22}{7}$. The absolute error is 

$$ E_{\mathrm{abs}} = \left\lvert \pi - \frac{22}{7} \right\rvert  \approx 0.0013 $$

Absolute errors are usually less useful than relative errors. If I am computing a distance and I find that my result is out by 15km, that might seem like I've made a terrible mistake. But if I was computing the distance of Earth from the sun, then 15km is a trivial difference (around 0.0001% off). This is way of computing the error is called the relative error, and is given by 

$$E_{\mathrm{rel}} = \frac{E_{\mathrm{abs}}}{\mathrm{Exact~value}}$$

which for the $\pi$ example is 

$$ E_{\mathrm{rel}} = \frac{0.0013}{\pi} \approx 0.0004$$

The relative error tells us roughly how many correct digits we have: in this case, the three leading zeros tell us the error is in the fourth significant figure.


In [ ]:
print(np.pi)
print(22/7)

---

## 2. Two Sources of Error

| Error type | Origin | How to reduce it |
|---|---|---|
| **Truncation error** | Replacing an infinite process (e.g. Taylor series) with a finite approximation | Use more terms, or smaller step size $h$ |
| **Rounding error** | Finite precision of floating-point arithmetic | Higher-precision types; reformulate the calculation |

These two sources **compete**: decreasing $h$ reduces truncation error but eventually
amplifies rounding error because we subtract nearly-equal numbers. There is an *optimal $h$*
that minimises total error — and finding it is a key to solving problems many problems in modelling.

---

## 3. Error example: Numerical Differentiation

The standard way we compute the derivative of a function requires we know what that function is. If I give you a function, say $f(x) = \sin(x)$, you know how to apply the rules of differentiation to compute $f'(x) = \cos(x)$.

Commonly, though, we might want $f'(x)$ but we don't know what $f(x)$ is: we might just have the output of a simulation, or a list of experimental measurements. For example, say you want to know what velocity a car is travelling at. You might know that the definition of velocity is the rate of change of position, $\frac{dr}{dt}$, but unfortunately, we don't know what $r(t)$ is. We might, however, be able to measure where the car is at a few, _discrete_ times: that's what a speed gun does:

| time (s)| position (m)| 
|---|---|
|0.0 | 0.0 |
|0.1 | 1 |
|0.2 | 4 | 
|0.3 | 9 |

And from that, we can work out the average speed in those time _intervals_:


| time (s)| speed (m/s)| 
|---|---|
|0.0 - 0.1| (1-0)/0.1 = 10 |
|0.1 - 0.2| (4-1)/0.1 = 30 |
|0.2 - 0.3| (9-4)/0.1 = 50| 

Whereas calculus requires _infinitessimally_ small intervals of time (i.e. continuous functions), what we are doing here uses *finite* intervals. Because the derivative requires the _difference_ between the function values at two instants of time, we call this method of differentation a **finite difference**. This can be formalised using the Taylor series:

$$f(x\pm h) = f(x) \pm h f'(x) + \frac{h^2}{2}f''(x) \pm \frac{h^3}{6}f'''(x) + \cdots$$

There are several ways we can find an expression for the derivative $f'(x)$ by rearranging these terms:

**Forward difference**
$$f'(x) \approx \frac{f(x+h) - f(x)}{h}$$
We arrive here by ignoring all the terms of order $h^2$ (we usually write $O(h^2)$: $O$ for 'order') and higher. We can _neglect_ those terms if they are _negligible_, which they will be, if $h$ is sufficiently small.

**Backward difference**:
$$f'(x) \approx \frac{f(x) - f(x-h)}{h}$$
As before, we ignore all the terms of $O(h^2)$ and higher. But notice that whereas in the _forward_ difference, the derivative at point $x$ was calculated using the function value at $x$ and one, finite step 'forward', this instead uses the function value at $x$ and one step 'backwards'. 

Both the forward and backward difference ignore terms of $O(h^2)$. Thus the _largest contribution to the error_ (i.e. all the stuff we ignore) is $O(h^2)$. We say that these methods are "_accurate to first order in_ $h$".

**Central difference** 
$$f'(x) \approx \frac{f(x+h) - f(x-h)}{2h}$$
We can obtain this formula by ignoring everything of $O(h^3)$ and higher, and then subtracting the expansions for $f(x+h)$ and $f(x-h)$ which cancels the $f''(x)$ terms. 

The central difference is *more accurate at the same cost* because its leading error term is $O(h^3)$. Practically, this means that if we have a fixed $h$, the 'second order', central difference method is more accurate. Alternatively, if we have control over $h$, we might choose the 'simpler' first order method. In practice, we usually have to balance these with [other considerations](https://en.wikipedia.org/wiki/Von_Neumann_stability_analysis).

<div style="background:#f7d6e3;border-left:4px solid #f32121;padding:16px;margin:16px 0;border-radius:4px;width:100%;box-sizing:border-box;overflow-wrap:break-word;">

# Exercise

Implement the backward and central difference formulas in the functions below. The forward difference is done for you. 

(Also, if you did this for MTH1025, note that we're handling things differently here. We're assuming that we know the function we're trying to differentiate – it's passed as an argument to the `forward_diff` function – so that we can compute the function values $f(x+h)$, $f(x)$ etc.)
</div>

In [ ]:
def forward_diff(f, x, h):
    """given a function (f), a value (x) and a grid spacing (h), compute the forward difference formula for f'(x)"""
    return (f(x+h) - f(x))/h

def backward_diff(f, x, h):
    return

def central_diff(f, x, h):
    return

In [ ]:
runtest(['test_backward', 'test_central'])

---

Now we'll test the methods, computing the derivative of a known function ($\sin(x)$) at a single point ($x=1$) via each method.

In [ ]:

f, df_exact = np.sin, np.cos
x0    = 1.0
exact = df_exact(x0)

print(f"Exact f'({x0}) = cos({x0}) = {exact}")
print()
for h in [0.1, 0.01, 0.001]:
    try:
        fd = forward_diff(f, x0, h)
        bd = backward_diff(f, x0, h)
        cd = central_diff(f, x0, h)
    except NameError: 
        fd, bd, cd = None, None, None
        pass
    if fd is not None and bd is not None and cd is not None:
        print(f"h = {h}:")
        print(f"  Forward  {fd}   error {abs(fd - exact):.2e}")
        print(f"  Backward {bd}   error {abs(bd - exact):.2e}")
        print(f"  Central  {cd}   error {abs(cd - exact):.2e}")
        print()
    else:
        print("backward and central difference not implemented yet")

<div style="background:#e8f4f8;border-left:4px solid #2196F3;padding:16px;margin:16px 0;border-radius:4px;width:100%;box-sizing:border-box;overflow-wrap:break-word;">

**What do you notice about the behaviour of the error for the central difference method when we make $h$ 10X smaller each time?**

<details style="margin: 1em 0;">
<summary style="cursor:pointer; font-weight:bold; color:#2196F3;">
 Reveal answer
</summary>
<div style="background:#f0f7ff; border-left:4px solid #2196F3; padding:12px 16px; margin-top:8px; border-radius:4px; box-sizing:border-box; overflow-wrap:break-word;">

While the error for the forward and backward method reduces by 10X (one order of magnitude) each time we reduce _h_ by 10X, the error for the central method reduces by 100X (two orders of magnitude)

</div>
</details>

</div>

---

<div style="background:#f7d6e3;border-left:4px solid #f32121;padding:16px;margin:16px 0;border-radius:4px;width:100%;box-sizing:border-box;overflow-wrap:break-word;">

# Exercise

Having implemented the forward, backward and central difference formulas, now we want to visualise their performance. 

1. Set up an array, `h_values` which contains grid space sizes for $h=[0.1, 0.001, 0.0001, ... 1e-15]$
2. For every value of `h` in that array, calculate the error of the forward, backward and central difference methods for computing the derivative of $\sin(x)$ at $x=1$. Save those values in the arrays `err_fwd`, `err_bwd` and `err_cen` respectively.
3. Plot these errors against `h_values` on a log-log plot. 
4. Label the x-axis 'Step size', and the y-axis 'Absolute Error'
</div>

In [ ]:
# your code goes here





# don't delete this line: it's needed for the Automated Feedback
fighand = plt.gca()


In [ ]:
runtest(['test_hvalues', 'test_err_fwd', 'test_err_bwd', 'test_err_cen', 'test_plot'])

<div style="background:#e8f4f8;border-left:4px solid #2196F3;padding:16px;margin:16px 0;border-radius:4px;width:100%;box-sizing:border-box;overflow-wrap:break-word;">

**The plot should show two distinct behaviours: one for larger values of $h$, and one as $h$ gets progressively smaller. Can you explain what's happening in terms of truncation and round-off error?**

<details style="margin: 1em 0;">
<summary style="cursor:pointer; font-weight:bold; color:#2196F3;">
 Reveal answer
</summary>
<div style="background:#f0f7ff; border-left:4px solid #2196F3; padding:12px 16px; margin-top:8px; border-radius:4px; box-sizing:border-box; overflow-wrap:break-word;">

For large values of $h$, the main source of error is truncation error: our method just isn't accurate enough. As we move (reading the axis from right to left) from larger to smaller values of $h$, the method gets more accurate (the error gets smaller). 

When $h$ gets small enough (for the central difference method, small enough is $10^{-5}$, for the forward/backward methods, it's around $10^{-8}$) the main source of error is round-off error. In other words, we're taking the difference between two numbers that are almost identical, and dividing by a number that is almost zero. The machine precision is not sufficiently small to give us accurate answers. As we move to smaller and smaller values of $h$, this error actually grows in size. 

</div>
</details>

</div>

<div style="background:#f7d6e3;border-left:4px solid #f32121;padding:16px;margin:16px 0;border-radius:4px;width:100%;box-sizing:border-box;overflow-wrap:break-word;">

# Exercise

**1. Second derivative.** The *second central difference* approximation is:
$$f''(x) \approx \frac{f(x+h) - 2f(x) + f(x-h)}{h^2} + O(h^2)$$
Implement this in a function, `second_diff`, and plot its error vs $h$ for the same values of $h$ we used before, and for $f(x) = e^x$ at $x = 1$.

What is the optimal step size? How does this compare to the first-derivative case?


<details style="margin: 1em 0;">
<summary style="cursor:pointer; font-weight:bold; color:#2196F3;">
 Reveal answer
</summary>
<div style="background:#f0f7ff; border-left:4px solid #2196F3; padding:12px 16px; margin-top:8px; border-radius:4px; box-sizing:border-box; overflow-wrap:break-word;">

You should notice that the optimal step size is a bit larger in this case: somewhere around $10^{-4}$. The reason is that the exponential function is reasonably 'flat' around $x=1$, so the top line of the central difference formula becomes very close to zero very quickly. Then, because we are now dividing by $h^2$, the round-off error grows as we divide by smaller and smaller numbers.

</div>
</details>

</div>
</div>



In [ ]:
# your code goes here



# don't delete this line: it's needed for the Automated Feedback
fighand = plt.gca()

In [ ]:
runtest(['test_2nd', 'test_plot2'])

**2. Modelling connection.** In Notebook 4 you will integrate an ODE numerically using step
size $h$. Think now about the trade-off: what goes wrong if $h$ is too large? What goes
wrong if $h$ is too small? Write a short note (5–8 sentences) predicting what you expect
to see before you run the experiments.